# 04 Data quality gate

Runs the rules in `nb_lib_retail.dq_rules` against bronze, silver and gold, and saves the
results to `meta.dq_results` (history) and `meta.dq_latest` (last run, used by the report).

If a **blocking** rule fails, the notebook raises an error, so the pipeline run fails and
nobody builds a decision on numbers that do not reconcile. **Warnings** are saved and shown
in the report but do not stop the run.

In [ ]:
run_date = ""
run_id = ""
fail_on_blocking = True

In [ ]:
import datetime as dt
import json

from pyspark.sql import functions as F

In [ ]:
%run nb_lib_retail

In [ ]:
spark.conf.set("spark.sql.session.timeZone", "UTC")
run_date = run_date or dt.date.today().isoformat()
run_id = run_id or dt.datetime.utcnow().strftime("manual-%Y%m%d%H%M%S")

t = {
    "bronze_pos": spark.table("bronze.pos_sales_raw"),
    "bronze_web": spark.table("bronze.web_orders_raw"),
    "silver_sales": spark.table("silver.sales_lines"),
    "quarantine": spark.table("silver.sales_quarantine"),
    "fact_sales": spark.table("gold.fact_sales"),
    "fact_target": spark.table("gold.fact_target"),
    "dim_store": spark.table("gold.dim_store"),
    "dim_product": spark.table("gold.dim_product"),
    "dim_date": spark.table("gold.dim_date"),
    "dim_city": spark.table("gold.dim_city"),
}
results = dq_rules(t, run_date)

In [ ]:
schema = ("rule_id string, table_name string, description string, severity string, "
          "failed_rows bigint, checked_rows bigint, status string")
df = (spark.createDataFrame([tuple(r[k] for k in ["rule_id", "table_name", "description", "severity",
                                                  "failed_rows", "checked_rows", "status"]) for r in results], schema)
      .withColumn("run_id", F.lit(run_id))
      .withColumn("run_date", F.lit(run_date).cast("date"))
      .withColumn("checked_at", F.current_timestamp()))
df.write.format("delta").mode("append").saveAsTable("meta.dq_results")
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("meta.dq_latest")
display(df.orderBy("rule_id"))

In [ ]:
blocking = [r for r in results if r["severity"] == "blocking" and r["status"] == "fail"]
warnings = [r for r in results if r["severity"] == "warning" and r["status"] == "fail"]
summary = {"run_id": run_id, "rules": len(results), "blocking_failed": len(blocking), "warnings": len(warnings)}
print(json.dumps(summary))
if blocking and fail_on_blocking:
    raise Exception("Data quality gate failed: " + "; ".join(f"{r['rule_id']} {r['description']} ({r['failed_rows']})" for r in blocking))